# ExternalPatch demo — Dimer formation

## Overview

**ExternalPatch** is a 4-body "external patch" force for HOOMD-blue.
Each patched particle *i* has a partner *j* that defines the patch direction
$\hat p = (\mathbf{r}_i - \mathbf{r}_j)/|\cdots|$ (pointing *outward*).
The interaction between two patched particles *i* and *k* is:

$$
U_{ik} = f_i\,f_k\;\varepsilon\!\left(1 - r^2/r_c^2\right)^2
$$

with cubic Hermite (smoothstep) angular envelopes
$f = S\!\left(\frac{\hat p \cdot \hat r - (1 - w)}{w}\right)$, where
$S(t) = 3t^2 - 2t^3$ for $0 \le t \le 1$.

A narrow patch (`width = 0.25`, transition from $u = 0.75$ to $1.0$,
≈ 41° half-angle) ensures each P particle can only
bind **one** partner → the dominant structure is a **dimer**.

All particles interact via DPD (conservative + dissipative + random)
for excluded volume and thermostating.

| Parameter | Value | Notes |
|-----------|-------|-------|
| N_dipoles | 1 000 | 2 000 particles total |
| Box | L × L × 5 | Flat 3D slab |
| ε | 30 | ~30 kT → stable dimers |
| width | 0.25 | Hermite transition width |
| r_cut (patch) | 1.5 | Patch attraction range |
| A (DPD) | 40 | Soft repulsion (P–P only) |
| γ (DPD) | 1.0 | DPD friction |
| r_cut (DPD) | 1.0 | DPD cutoff |
| Bond k | 200 | Stiff P–D tether |
| Bond r₀ | 0.5 | Short director |
| kT | 1.0 | Thermal energy |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import hoomd
from hoomd import align_angle

print("HOOMD version:", hoomd.version.version)


## 1. Build the initial configuration

Place 1 000 dipoles on a jittered grid in a flat 3D box (Lz = 5).
Each dipole: particle 2i is **P** (patched), particle 2i+1 is **D** (director).
Random 3D patch orientations.


In [ ]:

# ── Parameters ────────────────────────────────────────────────────────────
N_dipoles = 1000
N = 2 * N_dipoles
bond_r0 = 0.5          # P–D bond length
Lz = 5.0               # thin slab thickness
grid_side = int(np.ceil(np.sqrt(N_dipoles)))  # ~32
spacing = 1.0          # grid spacing
L = grid_side * spacing + 4.0  # box side (x and y)

device = hoomd.device.auto_select()
snap = hoomd.Snapshot(device.communicator)

if snap.communicator.rank == 0:
    snap.configuration.box = [L, L, Lz, 0, 0, 0]  # flat 3D box
    snap.particles.N = N
    snap.particles.types = ["P", "D"]
    snap.particles.mass[:] = 1.0

    rng = np.random.default_rng(42)

    positions = np.zeros((N, 3))
    typeids = np.zeros(N, dtype=int)

    for idx in range(N_dipoles):
        row = idx // grid_side
        col = idx % grid_side
        # Jittered grid position for the P particle
        cx = (col - grid_side / 2) * spacing + rng.uniform(-0.3, 0.3)
        cy = (row - grid_side / 2) * spacing + rng.uniform(-0.3, 0.3)
        cz = rng.uniform(-Lz / 2 + 0.5, Lz / 2 - 0.5)

        # Random 3D patch direction (uniform on sphere)
        cos_th = rng.uniform(-1, 1)
        sin_th = np.sqrt(1 - cos_th**2)
        phi = rng.uniform(0, 2 * np.pi)
        dx = bond_r0 * sin_th * np.cos(phi)
        dy = bond_r0 * sin_th * np.sin(phi)
        dz = bond_r0 * cos_th

        i_P = 2 * idx
        i_D = 2 * idx + 1

        positions[i_P] = [cx, cy, cz]
        positions[i_D] = [cx + dx, cy + dy, cz + dz]
        typeids[i_P] = 0   # P
        typeids[i_D] = 1   # D

    snap.particles.position[:] = positions
    snap.particles.typeid[:] = typeids
    snap.particles.velocity[:] = rng.normal(0, 0.5, (N, 3))

    # Bonds
    snap.bonds.N = N_dipoles
    snap.bonds.types = ["PD"]
    snap.bonds.typeid[:] = 0
    for idx in range(N_dipoles):
        snap.bonds.group[idx] = [2 * idx, 2 * idx + 1]

print(f"N_dipoles = {N_dipoles}, N_particles = {N}, L = {L:.1f}, Lz = {Lz:.1f}")


## 2. Set up forces and integrator

In [ ]:
sim = hoomd.Simulation(device=device, seed=42)
sim.create_state_from_snapshot(snap)

# ── DPD (conservative + dissipative + random, thermostat included) ────────
nlist_dpd = hoomd.md.nlist.Cell(buffer=0.4)
dpd = hoomd.md.pair.DPD(nlist=nlist_dpd, default_r_cut=1.0, kT=1.0)
dpd.params[("P", "P")] = dict(A=40.0, gamma=1.0)
dpd.params[("P", "D")] = dict(A=0.0, gamma=1.0)   # D is a phantom direction marker
dpd.params[("D", "D")] = dict(A=0.0, gamma=1.0)

# ── Harmonic bond (P ↔ D tether) ─────────────────────────────────────────
bond = hoomd.md.bond.Harmonic()
bond.params["PD"] = dict(k=200.0, r0=bond_r0)

# ── ExternalPatch (patch attraction between P particles) ──────────────────
nlist_patch = hoomd.md.nlist.Cell(buffer=0.4)
patch = align_angle.ExternalPatch(nlist=nlist_patch, r_cut=1.5)
patch.epsilon = 30.0
patch.width   = 0.25
patch.partners = [(2 * i, 2 * i + 1) for i in range(N_dipoles)]

# ── NVE integrator (DPD provides the thermostat) ─────────────────────────
method = hoomd.md.methods.ConstantVolume(filter=hoomd.filter.All())

integrator = hoomd.md.Integrator(dt=0.005, forces=[dpd, bond, patch],
                                  methods=[method])
sim.operations.integrator = integrator

thermo = hoomd.md.compute.ThermodynamicQuantities(filter=hoomd.filter.All())
sim.operations.computes.append(thermo)

print("Forces: DPD (P-P A=40, rc=1.0), Harmonic bond (k=200, r0=0.5), ExternalPatch (ε=30, width=0.25)")

## 3. Equilibrate and save snapshots

Run 500 000 steps (2 500 τ) total.  Save snapshots at the start,
midpoint, and end.

In [ ]:

snapshots = {}

# Initial state
sim.run(0)
snapshots["t = 0"] = sim.state.get_snapshot()
ke_per_particle = thermo.kinetic_energy / N
print(f"t = 0:     PE = {thermo.potential_energy:.1f},  KE/particle = {ke_per_particle:.3f}")

# Midpoint
sim.run(250_000)
snapshots["t = 1250 τ"] = sim.state.get_snapshot()
ke_per_particle = thermo.kinetic_energy / N
print(f"t = 1250τ: PE = {thermo.potential_energy:.1f},  KE/particle = {ke_per_particle:.3f}")

# Final
sim.run(250_000)
snapshots["t = 2500 τ"] = sim.state.get_snapshot()
ke_per_particle = thermo.kinetic_energy / N
print(f"t = 2500τ: PE = {thermo.potential_energy:.1f},  KE/particle = {ke_per_particle:.3f}")



## 4. Visualize dimer formation

The box is thin (Lz = 5) so an **x–y projection** shows the structure
clearly.  Each panel shows:
- **Green** circles: P particles in a dimer (3D NN distance < 1.3)
- **Red** circles: free P particles
- **Orange** dots: D (director) particles — define patch directions
- **Green** lines: dimer bonds projected onto x–y

A **zoomed-in** panel shows a random 8 × 8 region of the final state
so individual dimers are clearly visible.


In [ ]:

def find_dimers(pos_P, box, cutoff=1.3):
    """Return list of (i, j) dimer pairs among P particles (brute-force, 3D PBC)."""
    n = len(pos_P)
    Lx, Ly, Lz = box
    dimers = []
    paired = set()
    cutsq = cutoff * cutoff
    for i in range(n):
        if i in paired:
            continue
        best_j, best_dsq = -1, cutsq
        for j in range(i + 1, n):
            if j in paired:
                continue
            dx = pos_P[j, 0] - pos_P[i, 0]
            dy = pos_P[j, 1] - pos_P[i, 1]
            dz = pos_P[j, 2] - pos_P[i, 2]
            dx -= Lx * np.round(dx / Lx)
            dy -= Ly * np.round(dy / Ly)
            dz -= Lz * np.round(dz / Lz)
            dsq = dx * dx + dy * dy + dz * dz
            if dsq < best_dsq:
                best_dsq = dsq
                best_j = j
        if best_j >= 0:
            dimers.append((i, best_j))
            paired.add(i)
            paired.add(best_j)
    return dimers, paired


def plot_state(ax, snapshot, title, box, cutoff=1.3, zoom=None):
    """Plot P and D particles (x-y projection) with dimer bonds.

    If zoom=(cx, cy, half_w) is given, only show that region.
    """
    pos = np.array(snapshot.particles.position)
    tid = np.array(snapshot.particles.typeid)
    Lx, Ly, Lz = box

    pos_P = pos[tid == 0]
    pos_D = pos[tid == 1]

    dimers, paired = find_dimers(pos_P, box, cutoff)

    # Sizes and alpha depend on zoom level
    if zoom is not None:
        s_P, s_D, lw_bond = 40, 15, 3.0
    else:
        s_P, s_D, lw_bond = 8, 3, 1.5

    # Color P particles
    colors = ["green" if i in paired else "red" for i in range(len(pos_P))]
    ax.scatter(pos_P[:, 0], pos_P[:, 1], c=colors, s=s_P, zorder=2,
               edgecolors="none")

    # P–D tether lines (show director attachment)
    patch_dir = pos_D - pos_P
    patch_dir[:, 0] -= Lx * np.round(patch_dir[:, 0] / Lx)
    patch_dir[:, 1] -= Ly * np.round(patch_dir[:, 1] / Ly)
    patch_dir[:, 2] -= Lz * np.round(patch_dir[:, 2] / Lz)

    # D particle positions (x-y projection)
    d_xy_x = pos_P[:, 0] + patch_dir[:, 0]
    d_xy_y = pos_P[:, 1] + patch_dir[:, 1]

    # Draw P→D dipole tether lines
    lw_tether = 1.5 if zoom is not None else 0.5
    for k in range(len(pos_P)):
        ax.plot([pos_P[k, 0], d_xy_x[k]],
                [pos_P[k, 1], d_xy_y[k]],
                color="orange", lw=lw_tether, alpha=0.6, zorder=1)

    ax.scatter(d_xy_x, d_xy_y, c="orange", s=s_D, zorder=2,
               edgecolors="none", alpha=0.7, label="D (director)")

    # Draw dimer bonds ON TOP (x-y projection)
    for i, j in dimers:
        dx = pos_P[j, 0] - pos_P[i, 0]
        dy = pos_P[j, 1] - pos_P[i, 1]
        dx -= Lx * np.round(dx / Lx)
        dy -= Ly * np.round(dy / Ly)
        ax.plot([pos_P[i, 0], pos_P[i, 0] + dx],
                [pos_P[i, 1], pos_P[i, 1] + dy],
                color="darkgreen", lw=lw_bond, alpha=0.85, zorder=3,
                solid_capstyle="round")

    frac = len(paired) / len(pos_P)
    ax.set_title(f"{title}\n{len(dimers)} dimers ({frac:.0%} paired)",
                 fontsize=11)

    if zoom is not None:
        cx, cy, hw = zoom
        ax.set_xlim(cx - hw, cx + hw)
        ax.set_ylim(cy - hw, cy + hw)
    else:
        half_x = Lx / 2
        half_y = Ly / 2
        ax.set_xlim(-half_x, half_x)
        ax.set_ylim(-half_y, half_y)
    ax.set_aspect("equal")
    ax.tick_params(labelsize=8)


box_dims = (L, L, Lz)

# ── Overview panels (3 time points) ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (label, snap_i) in zip(axes, snapshots.items()):
    plot_state(ax, snap_i, label, box_dims)

fig.suptitle("Dimer formation in a patchy-dipole gas (x–y projection)",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# ── Zoomed-in panel (random 8×8 region of the final state) ───────────────
rng_zoom = np.random.default_rng(123)
half_box = L / 2
zoom_hw = 4.0  # half-width of zoom region
cx = rng_zoom.uniform(-half_box + zoom_hw, half_box - zoom_hw)
cy = rng_zoom.uniform(-half_box + zoom_hw, half_box - zoom_hw)

fig_z, ax_z = plt.subplots(figsize=(7, 7))
plot_state(ax_z, snapshots["t = 2500 τ"], "Zoomed in (final state)", box_dims,
           zoom=(cx, cy, zoom_hw))
ax_z.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()


## 5. Nearest-neighbor distance histogram

A clear peak near r ≈ 0.9–1.0 (contact distance) with a gap before the
bulk background confirms dimers are the dominant structure.

In [ ]:

# Compute all P–P nearest-neighbor distances at final state (3D PBC)
final_snap = snapshots["t = 2500 τ"]
pos = np.array(final_snap.particles.position)
tid = np.array(final_snap.particles.typeid)
pos_P = pos[tid == 0]
n = len(pos_P)

nn_dists = []
for i in range(n):
    best_dsq = np.inf
    for j in range(n):
        if j == i:
            continue
        dx = pos_P[j, 0] - pos_P[i, 0]
        dy = pos_P[j, 1] - pos_P[i, 1]
        dz = pos_P[j, 2] - pos_P[i, 2]
        dx -= L * np.round(dx / L)
        dy -= L * np.round(dy / L)
        dz -= Lz * np.round(dz / Lz)
        dsq = dx * dx + dy * dy + dz * dz
        if dsq < best_dsq:
            best_dsq = dsq
    nn_dists.append(np.sqrt(best_dsq))

nn_dists = np.array(nn_dists)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(nn_dists, bins=60, range=(0, 4), color="steelblue", edgecolor="white",
        linewidth=0.3)
ax.axvline(1.3, color="red", ls="--", lw=1, label="dimer cutoff (1.3)")
ax.set_xlabel("Nearest P–P distance", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("NN distance distribution (final state)", fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

dimer_frac = np.mean(nn_dists < 1.3)
print(f"Fraction of P particles with NN < 1.3: {dimer_frac:.1%}")


## Summary

With a patch half-angle **α = 0.5 rad ≈ 29°** and attraction strength
**ε = 30 kT**, nearly all patched particles find a dimer partner
inside a flat 3D slab (Lz = 5).
The narrow patch ensures each particle can only bind one partner at
a time — trimers or larger clusters are geometrically forbidden.

**To experiment:**
- Widen α → 1.0 to allow trimers / small clusters
- Lower ε → 2 to see frequent dimer breaking / re-forming
- Increase density (decrease L) for faster kinetics